# CourtListener search → DataFrame (HCDE 530) — **`week5_courtlistener_cases.ipynb`**

Uses the **local CourtListener CSV export** (no API token or live calls).

**File:** `MiniProject1/data/courtlistener.csv` or `week 6/Week6_files/courtlistener_week5_repull_40k.csv`.

Builds three slices (sample opinions, Seattle federal, newest King County appellate), then **Part 2** stacks them for **pandas** (`head` / `info`, `value_counts`, filter, `groupby` + `mean`, `isnull`).


## Part 2 — Pandas analysis (MP1-style)

**Five in-class operations are in this notebook**

| Operation | Question it answers | Where / column |
|-----------|---------------------|----------------|
| `df.head()` and `df.info()` | What does the data look like? dtypes and non-null counts? | Stacked `df_mp1` |
| `df['column'].value_counts()` | What judge strings appear most in the Seattle federal sample? | `df_jurisdiction['judge']` |
| `df[df['column'] >= value]` | Filter to decisions on or after a cutoff (here: 1990-01-01) | `df_mp1['_decision_dt']` vs `pd.Timestamp('1990-01-01')` |
| `df.groupby('column')['other'].mean()` | Mean `cite_count` by dataset slice after filtering | `df_recent_window.groupby('dataset')['cite_count']` |
| `df.isnull().sum()` | Missing cells per column across the stacked frame | `df_mp1` |


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def _ensure_loader() -> None:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        w6 = base / "week 6" / "Week6_files"
        if (w6 / "courtlistener_csv_loader.py").is_file():
            p = str(w6)
            if p not in sys.path:
                sys.path.insert(0, p)
            return
    raise FileNotFoundError("week 6/Week6_files/courtlistener_csv_loader.py not found.")


_ensure_loader()
from courtlistener_csv_loader import (
    load_courtlistener_csv,
    slice_king_county_recent,
    slice_sample_opinions,
    slice_seattle_federal,
    split_parties,
)

master = load_courtlistener_csv()
sample = slice_sample_opinions(master, n=25)
print("sample opinions (party-parseable):", len(sample))


Loaded 39,040 rows from /Users/rnr6/Documents/HCDE/hcde530/MiniProject1/data/courtlistener.csv
sample opinions (party-parseable): 25


In [2]:
rows: list[dict[str, object]] = []
for _, r in sample.iterrows():
    plaintiff, defendant = split_parties(str(r["case"]))
    rows.append(
        {
            "case": r["case"],
            "plaintiff": plaintiff or pd.NA,
            "defendant": defendant or pd.NA,
            "judge": r["judge"] or pd.NA,
            "date_of_decision": r["date_of_decision"],
            "cite_count": int(r["cite_count"]) if pd.notna(r["cite_count"]) else 0,
        }
    )

df = pd.DataFrame(rows)
df


,case,plaintiff,defendant,judge,date_of_decision,cite_count
0,"State Of Washington, V. Justin R. Smith","State Of Washington,",Justin R. Smith,NaN,2025-07-21,0
1,"Chen Wang And Liyin Xue, V. Dongmei Huang","Chen Wang And Liyin Xue,",Dongmei Huang,NaN,2025-08-25,0
2,"Erwin Chappel, Respondent/cr-appellants V. Dou...","Erwin Chappel, Respondent/cr-appellants","Douglas Johnson, Appellant/cr-respondents",NaN,2025-09-15,0
3,"Alaska Airlines, V. Hillary Spanjer","Alaska Airlines,",Hillary Spanjer,NaN,2025-09-15,0
4,"Samantha Snodderly, V. Bradley Shockey","Samantha Snodderly,",Bradley Shockey,NaN,2025-09-22,0
5,"State Of Washington, V. Kimcha Chhim","State Of Washington,",Kimcha Chhim,NaN,2025-08-19,0
6,"Clifton A. Little Ii Et Ano, V. Hardie-tynes C...","Clifton A. Little Ii Et Ano,",Hardie-tynes Co. Inc.,NaN,2025-08-25,0
7,State of Washington v. Christopher Floe,State of Washington,Christopher Floe,NaN,2025-07-29,0
8,"Lisa Earl, V. City Of Tacoma, Scott Campbell","Lisa Earl,","City Of Tacoma, Scott Campbell",NaN,2025-06-17,0
9,"State Of Washington, V. Karen K. Peterson","State Of Washington,",Karen K. Peterson,NaN,2025-08-04,0


## Judgments classified by Judge names and jurisdiction

Federal **Seattle** area opinions use CourtListener’s **`court_id:wawd`** (U.S. District Court, **Western District of Washington**). Each row sets **`jurisdiction`** to the label **`Seattle jurisdiction`**; **`court`** is the API’s full court name.

## Most recent judgments (King County)

CourtListener does **not** give King County **Superior** Court its own `court_id`. To capture **King County** decisions that are published in the corpus, this query uses the **Court of Appeals of Washington** (`court_id:washctapp`) plus the phrase **`"King County"`** (appeals from King County Superior Court and other King County matters often appear here).

The API returns pages in relevance order, so the code **follows `next`**, collects results, **sorts by `dateFiled` descending**, and keeps the **top 20** most recent opinions.

In [3]:
from courtlistener_csv_loader import SEATTLE_JURISDICTION_LABEL

kc = slice_king_county_recent(master, n=20)
print("King County appellate rows (newest 20 in CSV):", len(kc))

rows_recent: list[dict[str, object]] = []
for _, r in kc.iterrows():
    rows_recent.append(
        {
            "case": r["case"],
            "judge": r["judge"] or pd.NA,
            "court": r["court"] or pd.NA,
            "court_id": r["court_id"],
            "date_filed": r["date_of_decision"],
            "cite_count": int(r["cite_count"]) if pd.notna(r["cite_count"]) else 0,
        }
    )

df_recent = pd.DataFrame(rows_recent)
df_recent


King County appellate rows (newest 20 in CSV): 20


,case,judge,court,court_id,date_filed,cite_count
0,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22,0
1,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22,0
2,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22,0
3,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22,0
4,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22,0
5,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22,0
6,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22,0
7,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22,0
8,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22,0
9,"Samantha Snodderly, V. Bradley Shockey",NaN,Court of Appeals of Washington,washctapp,2025-09-22,0


In [4]:
wawd = slice_seattle_federal(master, n=20)
print("Seattle-area (W.D. Wash.) rows used:", len(wawd))

rows_jurisdiction: list[dict[str, object]] = []
for _, r in wawd.iterrows():
    rows_jurisdiction.append(
        {
            "judge": r["judge"] or pd.NA,
            "jurisdiction": SEATTLE_JURISDICTION_LABEL,
            "court": r["court"] or pd.NA,
            "court_id": r["court_id"],
            "case": r["case"],
            "date_of_decision": r["date_of_decision"],
            "cite_count": int(r["cite_count"]) if pd.notna(r["cite_count"]) else 0,
        }
    )

df_jurisdiction = pd.DataFrame(rows_jurisdiction)
df_jurisdiction


Seattle-area (W.D. Wash.) rows used: 20


,judge,jurisdiction,court,court_id,case,date_of_decision,cite_count
0,NaN,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"Simmons v. Safeway, Inc.",2019-08-01,0
1,Settle,Seattle jurisdiction,"District Court, W.D. Washington",wawd,State v. Franciscan Health Sys.,2019-03-01,1
2,Leighton,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"Animal Legal Def. Fund v. Olympic Game Farm, Inc.",2019-05-21,3
3,Jones,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"Beane v. RPW Legal Servs., PLLC",2019-05-06,2
4,Lasnik,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Galvez v. Cuccinelli,2019-07-17,4
5,Leighton,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Mitchell v. Atkins,2019-05-20,1
6,Robart,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Calderon-Rodriguez v. Wilcox,2019-02-06,1
7,Pechman,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Gallupe v. Sedgwick Claims Mgmt. Servs. Inc.,2019-02-14,3
8,Lasnik,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"United Statesi Ins. Servs. Nat'l, Inc. v. Ogden",2019-03-06,1
9,Leighton,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Marine Carpenters Pension Fund v. Puglia Marin...,2019-04-10,0


### Question 1 — What does the combined CourtListener dataset look like?

We stack the three pulls into **`df_mp1`** so one frame carries Miranda search rows, W.D. Wash. rows, and King County appellate rows. **`head()`** shows example values; **`info()`** shows dtypes and how many non-null values exist per column.

In [5]:
# Question 1: After stacking all three slices, what do the first rows look like and what dtypes / non-null counts do we have?
# Answer meaning: head() shows real rows so we can sanity-check names and dates; info() tells us which columns are sparse or object-typed before we filter or group.

df_m = df.assign(dataset="Sample opinions (local CSV)")
df_j = df_jurisdiction.assign(dataset="W.D. Wash. (Seattle)")
df_r = df_recent.rename(columns={"date_filed": "date_of_decision"}).assign(
    dataset="King County (Wash. Ct. App.)"
)
df_mp1 = pd.concat([df_m, df_j, df_r], ignore_index=True, sort=False)

display(df_mp1.head(10))
df_mp1.info()


,case,plaintiff,defendant,judge,date_of_decision,cite_count,dataset,jurisdiction,court,court_id
0,"State Of Washington, V. Justin R. Smith","State Of Washington,",Justin R. Smith,NaN,2025-07-21,0,Sample opinions (local CSV),NaN,NaN,NaN
1,"Chen Wang And Liyin Xue, V. Dongmei Huang","Chen Wang And Liyin Xue,",Dongmei Huang,NaN,2025-08-25,0,Sample opinions (local CSV),NaN,NaN,NaN
2,"Erwin Chappel, Respondent/cr-appellants V. Dou...","Erwin Chappel, Respondent/cr-appellants","Douglas Johnson, Appellant/cr-respondents",NaN,2025-09-15,0,Sample opinions (local CSV),NaN,NaN,NaN
3,"Alaska Airlines, V. Hillary Spanjer","Alaska Airlines,",Hillary Spanjer,NaN,2025-09-15,0,Sample opinions (local CSV),NaN,NaN,NaN
4,"Samantha Snodderly, V. Bradley Shockey","Samantha Snodderly,",Bradley Shockey,NaN,2025-09-22,0,Sample opinions (local CSV),NaN,NaN,NaN
5,"State Of Washington, V. Kimcha Chhim","State Of Washington,",Kimcha Chhim,NaN,2025-08-19,0,Sample opinions (local CSV),NaN,NaN,NaN
6,"Clifton A. Little Ii Et Ano, V. Hardie-tynes C...","Clifton A. Little Ii Et Ano,",Hardie-tynes Co. Inc.,NaN,2025-08-25,0,Sample opinions (local CSV),NaN,NaN,NaN
7,State of Washington v. Christopher Floe,State of Washington,Christopher Floe,NaN,2025-07-29,0,Sample opinions (local CSV),NaN,NaN,NaN
8,"Lisa Earl, V. City Of Tacoma, Scott Campbell","Lisa Earl,","City Of Tacoma, Scott Campbell",NaN,2025-06-17,0,Sample opinions (local CSV),NaN,NaN,NaN
9,"State Of Washington, V. Karen K. Peterson","State Of Washington,",Karen K. Peterson,NaN,2025-08-04,0,Sample opinions (local CSV),NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 65 entries, 0 to 64
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   case              65 non-null     str           
 1   plaintiff         25 non-null     str           
 2   defendant         25 non-null     str           
 3   judge             19 non-null     object        
 4   date_of_decision  65 non-null     datetime64[us]
 5   cite_count        65 non-null     int64         
 6   dataset           65 non-null     str           
 7   jurisdiction      20 non-null     str           
 8   court             40 non-null     str           
 9   court_id          40 non-null     str           
dtypes: datetime64[us](1), int64(1), object(1), str(7)
memory usage: 5.2+ KB


### Question 2 — Which judge labels are most common in the Seattle federal sample?

The W.D. Wash. table keeps one **judge** string per opinion (sometimes empty). **`value_counts()`** ranks those labels so we see concentration versus variety on this page of results.

In [6]:
# Question 2: In the Seattle-area federal pull, which judge field strings show up most often on this page?
# Answer meaning: value_counts() tells us whether a few last names dominate the relevance-ranked first page or whether the mix is flat — it describes the *sample*, not who is “most important” statewide.

df_jurisdiction["judge"].value_counts(dropna=False)

judge
Settle                    4
Leighton                  3
Lasnik                    2
Robart                    2
Pechman                   2
Martinez                  2
NaN                       1
Jones                     1
Introduction, Leighton    1
Theiler                   1
Zilly                     1
Name: count, dtype: int64

### Question 3 — How sparse is the data, and how do post-1990 citation averages compare by slice?

**`isnull().sum()`** counts missing cells per column. A **boolean filter** keeps decisions on or after 1990 so older Miranda noise does not dominate. **`groupby(...).mean()`** on **`cite_count`** compares average indexed citations across the three pulls.

In [7]:
# Question 3a: Across all stacked rows, how many missing values does each column have?
# Answer meaning: isnull().sum() is a completeness audit — large counts (e.g., judge) warn us that person-based charts would be biased or empty for many rows.
missing_per_col = df_mp1.isnull().sum()
display(missing_per_col)

# Question 3b: Which opinions have a decision date on or after 1990-01-01?
# Answer meaning: filtering to a recent window keeps the citation comparison from being dragged down by decades-old Miranda hits that legitimately have different citation patterns.
df_mp1["_decision_dt"] = pd.to_datetime(df_mp1["date_of_decision"], errors="coerce")
df_recent_window = df_mp1[df_mp1["_decision_dt"] >= pd.Timestamp("1990-01-01")]
print("rows with decision date >= 1990-01-01:", len(df_recent_window))
display(df_recent_window.head(6))

# Question 3c: Inside that window, what is the mean cite_count for each dataset label?
# Answer meaning: groupby(dataset)['cite_count'].mean() summarizes CourtListener's citeCount field — low averages mean few later cases cite these opinions in the index, not that the underlying law is unimportant.
df_recent_window.groupby("dataset", dropna=False)["cite_count"].mean().round(2)

case                 0
plaintiff           40
defendant           40
judge               46
date_of_decision     0
cite_count           0
dataset              0
jurisdiction        45
court               25
court_id            25
dtype: int64

rows with decision date >= 1990-01-01: 65


,case,plaintiff,defendant,judge,date_of_decision,cite_count,dataset,jurisdiction,court,court_id,_decision_dt
0,"State Of Washington, V. Justin R. Smith","State Of Washington,",Justin R. Smith,NaN,2025-07-21,0,Sample opinions (local CSV),NaN,NaN,NaN,2025-07-21
1,"Chen Wang And Liyin Xue, V. Dongmei Huang","Chen Wang And Liyin Xue,",Dongmei Huang,NaN,2025-08-25,0,Sample opinions (local CSV),NaN,NaN,NaN,2025-08-25
2,"Erwin Chappel, Respondent/cr-appellants V. Dou...","Erwin Chappel, Respondent/cr-appellants","Douglas Johnson, Appellant/cr-respondents",NaN,2025-09-15,0,Sample opinions (local CSV),NaN,NaN,NaN,2025-09-15
3,"Alaska Airlines, V. Hillary Spanjer","Alaska Airlines,",Hillary Spanjer,NaN,2025-09-15,0,Sample opinions (local CSV),NaN,NaN,NaN,2025-09-15
4,"Samantha Snodderly, V. Bradley Shockey","Samantha Snodderly,",Bradley Shockey,NaN,2025-09-22,0,Sample opinions (local CSV),NaN,NaN,NaN,2025-09-22
5,"State Of Washington, V. Kimcha Chhim","State Of Washington,",Kimcha Chhim,NaN,2025-08-19,0,Sample opinions (local CSV),NaN,NaN,NaN,2025-08-19


dataset
King County (Wash. Ct. App.)    0.00
Sample opinions (local CSV)     0.00
W.D. Wash. (Seattle)            1.95
Name: cite_count, dtype: float64

### Notes

- **Data source**: local CSV only — no token required.
- **Stacked frame**: `df_mp1` combines sample + Seattle federal + newest King County appellate rows from the export.
